In [ ]:
# === Cell 1/4: GPU check + install deps ===
# Deliberately standalone -- no hippovoice repo clone needed for this
# sanity check, just Qwen2.5-Omni's own real dependencies plus pyttsx3
# for a single real spoken test question (a real audio file, not a
# synthetic silence stand-in, since the whole point is testing genuine
# audio comprehension).
import torch, subprocess, sys

assert torch.cuda.is_available(), (
    'No GPU detected -- set Accelerator to "GPU T4 x2" in Kaggle Settings '
    'before running anything below.'
)
print(f'GPU OK: {torch.cuda.get_device_name(0)}')
print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f'Command failed: {cmd}')

# espeak needed by pyttsx3 on Linux -- confirmed missing on Kaggle's base
# image in this project's other notebooks.
run('apt-get -qq install -y espeak-ng espeak > /dev/null 2>&1')
run('pip install -q pyttsx3 soundfile')

# Qwen2.5-Omni needs a specific transformers preview branch, not mainline --
# confirmed from the model's own HF card, not guessed.
run('pip install -q git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview')
run('pip install -q accelerate "qwen-omni-utils[decord]"')

print('Setup done.')


In [ ]:
# === Cell 2/4: Load Qwen2.5-Omni-3B ===
# 3B, not 7B: the 7B card's own stated VRAM figures (31GB for a 15s
# *video* clip in bf16) are for video+audio combined, heavier than our
# audio-only short-turn use case -- but a T4 here has ~14.5GB usable
# (confirmed in this project's other real OOM errors), and 7B's plain
# weights alone in bf16 are already ~14-15GB with zero headroom left for
# activations. 3B's weights (~6GB) leave real room to actually find out
# if this fits, rather than gambling the whole test on 7B's edge case.
import time
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor

t0 = time.perf_counter()
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-Omni-3B",
    torch_dtype="auto",
    device_map="auto",
)
processor = Qwen2_5OmniProcessor.from_pretrained("Qwen/Qwen2.5-Omni-3B")
load_time = time.perf_counter() - t0

print(f'Loaded in {load_time:.1f}s')
print(f'VRAM allocated after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print(f'VRAM reserved after load:  {torch.cuda.memory_reserved() / 1e9:.2f} GB')


In [ ]:
# === Cell 3/4: Synthesize one real spoken test question ===
# A real audio file with a genuinely checkable answer (mirrors the
# WeightEdit sanity check's own "Eiffel Tower -> Paris" pattern) --
# testing real audio comprehension, not a synthetic silence stand-in.
# Single pyttsx3 call in this whole script -- no deadlock risk (that bug
# needs a SECOND call in one process; see BUGS.md / tts/model.py).
import pyttsx3

QUESTION = "What is the capital of France? Answer in one word."
AUDIO_PATH = "/kaggle/working/question.wav"

engine = pyttsx3.init()
engine.save_to_file(QUESTION, AUDIO_PATH)
engine.runAndWait()

import os
print(f'Saved test question audio: {AUDIO_PATH} ({os.path.getsize(AUDIO_PATH)} bytes)')


In [ ]:
# === Cell 4/4: One real audio-in/audio-out turn, timed ===
# The actual question this whole notebook exists to answer: does
# model.generate() -- a genuine synchronous call, confirmed from Qwen's
# own HF card, not a continuous/full-duplex stream like Moshi or Nova
# Sonic turned out to be -- return a real, correct, complete answer, and
# how long does that one real call actually take on a Kaggle T4.
import time
import torch
import soundfile as sf
from qwen_omni_utils import process_mm_info

conversation = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant that answers questions concisely."}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": AUDIO_PATH},
        ],
    },
]

text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
inputs = processor(text=text, audio=audios, images=images, videos=videos,
                    return_tensors="pt", padding=True, use_audio_in_video=False)
inputs = inputs.to(model.device).to(model.dtype)

torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
text_ids, audio = model.generate(**inputs, use_audio_in_video=False, speaker="Chelsie")
elapsed = time.perf_counter() - t0

response_text = processor.batch_decode(text_ids, skip_special_tokens=True)[0]
peak_vram = torch.cuda.max_memory_allocated() / 1e9

OUTPUT_PATH = "/kaggle/working/response.wav"
sf.write(OUTPUT_PATH, audio.reshape(-1).detach().cpu().numpy(), samplerate=24000)

print('=' * 60)
print(f'Question audio:   {AUDIO_PATH}')
print(f'Response text:    {response_text!r}')
print(f'Response audio:   {OUTPUT_PATH}')
print(f'Wall-clock time:  {elapsed:.2f}s')
print(f'Peak VRAM used:   {peak_vram:.2f} GB')
print('=' * 60)
print(f'Correctness check (contains "Paris"): {"PASS" if "paris" in response_text.lower() else "FAIL"}')
print(f'Speed vs Gemini Live (35-45s/question, confirmed real): '
      f'{"FASTER" if elapsed < 35 else "NOT FASTER"} ({elapsed:.1f}s vs 35-45s)')

import json
with open('/kaggle/working/qwen_omni_sanity_result.json', 'w') as f:
    json.dump({
        'model': 'Qwen/Qwen2.5-Omni-3B',
        'gpu': torch.cuda.get_device_name(0),
        'response_text': response_text,
        'elapsed_seconds': elapsed,
        'peak_vram_gb': peak_vram,
        'correctness_pass': 'paris' in response_text.lower(),
    }, f, indent=2)
print("Saved: /kaggle/working/qwen_omni_sanity_result.json")
